# 04. Classification: kNN vs CART vs RF vs XGBoost

Train all 4 classifiers, tune with `RandomizedSearchCV`, and compare.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np


In [ ]:
from src.utils.config import load_config
from src.utils.io import load_parquet
from src.models.classification.train import ClassifierTrainer

cfg = load_config()
df = load_parquet(f"{cfg['paths']['data_processed']}/properties_features.parquet")
print(df.shape)

## Train all 4 classifiers

In [ ]:
trainer = ClassifierTrainer(cfg)
comparison, fitted = trainer.train_all(df, tune=True)
comparison

## Visualize comparison

In [ ]:
from src.visualization.model_plots import plot_classifier_comparison
plot_classifier_comparison(comparison, 'reports/figures/classifier_comparison.png')

## Confusion matrices for each model

In [ ]:
from src.utils.io import load_pickle
from src.visualization.model_plots import plot_confusion_matrix

results = load_pickle('models/classifier_results.pkl')['results']
for r in results:
    plot_confusion_matrix(r.confusion_matrix,
                          ['undervalued','fairly_valued','overvalued'],
                          f'{r.name} confusion matrix',
                          f'reports/figures/cm_{r.name}.png')
    print(r.classification_report)
    print('---')

## Feature importance

In [ ]:
from src.visualization.model_plots import plot_feature_importance

best = comparison.index[0]
clf = fitted[best]
feat_names = clf.pipeline.named_steps['preprocessor'].get_feature_names_out()
plot_feature_importance(clf.pipeline, list(feat_names), top_n=20,
                        save_path='reports/figures/feature_importance.png')

In [ ]:
import shap

best_clf = fitted[comparison.index[0]]
explainer = shap.TreeExplainer(best_clf.pipeline.named_steps['classifier'])

# Transform a sample through the preprocessor
X_sample = X_test.head(500)
X_transformed = best_clf.pipeline.named_steps['preprocessor'].transform(X_sample)
feat_names = best_clf.pipeline.named_steps['preprocessor'].get_feature_names_out()

shap_values = explainer.shap_values(X_transformed)
# For multi-class, shap_values is a list [class0, class1, class2]
shap.summary_plot(shap_values[0], X_transformed, feature_names=feat_names, max_display=15)

## SHAP Explainability (XGBoost — best model)
import shap

best_name = comparison.index[0]   # 'xgboost'
best_clf = fitted[best_name]

# Get the trained classifier out of the sklearn Pipeline
classifier = best_clf.pipeline.named_steps['classifier']
preprocessor = best_clf.pipeline.named_steps['preprocessor']

# Transform test set through preprocessor
X_test_transformed = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

# Create SHAP explainer
explainer = shap.TreeExplainer(classifier)
shap_values = explainer.shap_values(X_test_transformed[:500])  # sample for speed

# Summary plot — shows which features matter most AND in which direction
# shap_values is a list [class0_array, class1_array, class2_array]
shap.summary_plot(
    shap_values[0],                    # class 0 = undervalued
    X_test_transformed[:500],
    feature_names=feature_names,
    max_display=15,
    plot_type="bar",
    show=False
)
plt.title("SHAP Feature Importance — 'Undervalued' class")
plt.tight_layout()
plt.savefig('reports/figures/shap_summary.png', dpi=110, bbox_inches='tight')
plt.show()

# Force plot — explain one individual prediction
shap.initjs()
shap.force_plot(
    explainer.expected_value[0],
    shap_values[0][0],
    X_test_transformed[0],
    feature_names=feature_names
)

In [ ]:
## Per-class ROC Curves
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

best_name = comparison.index[0]
best_clf = fitted[best_name]
y_score = best_clf.predict_proba(X_test)

# Binarize labels for OvR
y_bin = label_binarize(y_test, classes=[0, 1, 2])
class_names = ['Undervalued', 'Fairly Valued', 'Overvalued']
colors = ['#43a047', '#1976d2', '#e53935']

fig, ax = plt.subplots(figsize=(9, 6))
for i, (name, color) in enumerate(zip(class_names, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'Per-class ROC Curves — {best_name}')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('reports/figures/roc_curves.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
## Learning Curve — bias/variance analysis
from sklearn.model_selection import learning_curve
import numpy as np

train_sizes, train_scores, val_scores = learning_curve(
    best_clf.pipeline,
    X_train, y_train,
    cv=3,
    train_sizes=np.linspace(0.1, 1.0, 8),
    scoring='f1_macro',
    n_jobs=2,
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='#1976d2', label='Training score')
ax.fill_between(train_sizes,
                train_scores.mean(axis=1) - train_scores.std(axis=1),
                train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.15, color='#1976d2')
ax.plot(train_sizes, val_scores.mean(axis=1), 'o-', color='#43a047', label='Cross-val score')
ax.fill_between(train_sizes,
                val_scores.mean(axis=1) - val_scores.std(axis=1),
                val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.15, color='#43a047')
ax.set_xlabel('Training set size')
ax.set_ylabel('F1 Macro')
ax.set_title(f'Learning Curve — {best_name}')
ax.legend()
plt.tight_layout()
plt.savefig('reports/figures/learning_curve.png', dpi=110, bbox_inches='tight')
plt.show()